<a href="https://colab.research.google.com/github/Physalis-Alkekengi/ASIGROMACS/blob/main/PreCharmmGUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# @title Ligand Topology Generation and Conformational Minimization Pipeline
import os
import subprocess
import sys

# Dependency check: RDKit required for cheminformatics/conformer generation
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem
    print("RDKit is ready.")
except ImportError:
    print("Installing RDKit...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rdkit"])
    from rdkit import Chem
    from rdkit.Chem import AllChem

# Substrate and Co-factor definitions
# Target: Oxidized PP Trimer (4-methylheptan-2-one derivative proxy)
PP_SMILES = "CC(C)CC(=O)CC(C)C"
# Co-factor: L-Glutathione (Reduced) for GST/Peroxidase coupling
GSH_SMILES = "C(CC(=O)N[C@@H](CS)C(=O)NCC(=O)O)[C@@H](C(=O)O)N"

# Ligand Prep Pipeline: SMILES -> 3D Coordinate Generation -> Energy Minimization
def build_polished_mol2(smiles, filename, resname):
    print(f"Building polished {resname} topology...")

    # Initialize graph representation and saturate valencies (Explicit H)
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)

    # Conformational Sampling (ETKDGv3 algorithm)
    # N=50 ensures coverage of rotamer space to avoid local minima
    print("Executing conformational sampling (N=50)...")
    cids = AllChem.EmbedMultipleConfs(mol, numConfs=50, params=AllChem.ETKDGv3())

    # Force Field Optimization (MMFF94) to relax steric clashes
    results = AllChem.MMFFOptimizeMoleculeConfs(mol, numThreads=0)

    # Convergence check and selection of global minimum (E_min)
    lowest_energy = float('inf')
    best_conf_id = -1

    for i, (not_conv, energy) in enumerate(results):
        if not_conv == 0 and energy < lowest_energy:
            lowest_energy = energy
            best_conf_id = cids[i]

    print(f"Global minimum identified (Energy: {lowest_energy:.2f})")

    # Serialize atom names (e.g., C1, C2) for forcefield topology compatibility
    atom_counts = {}
    for atom in mol.GetAtoms():
        sym = atom.GetSymbol()
        atom_counts[sym] = atom_counts.get(sym, 0) + 1
        atom.SetProp("_Name", f"{sym}{atom_counts[sym]}")

    # Write optimized geometry to temp PDB
    temp_pdb = f"temp_{resname}.pdb"
    Chem.MolToPDBFile(mol, temp_pdb, confId=best_conf_id)

    # Invoke OpenBabel for final conversion
    # Ensures Gasteiger partial charges are assigned for docking
    if not os.path.exists("/usr/bin/obabel"):
        subprocess.run("apt-get install -y openbabel > /dev/null", shell=True)

    cmd = f"obabel -ipdb {temp_pdb} -omol2 -O {filename} --unique --gen3d"
    subprocess.run(cmd, shell=True, check=True)

    if os.path.exists(temp_pdb): os.remove(temp_pdb)
    print(f"Geometry optimization complete: {filename}")

# Batch execution for docking ligands
build_polished_mol2(PP_SMILES, "plastic_final.mol2", "I01")
build_polished_mol2(GSH_SMILES, "gsh_final.mol2", "GSH")

print("\nOutput Generation Complete. Files ready for download:")
print("   1. plastic_final.mol2 (Upload this to I01)")
print("   2. gsh_final.mol2 (Upload this to GSH)")

In [ ]:
# @title Structure Assembly and Topology Generation Pipeline (Protenix Target)
import os
import subprocess
import sys
import numpy as np

# Dependency check: RDKit (Cheminformatics) and MDAnalysis (Trajectory/Structure manipulation)
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem
    import MDAnalysis as mda
    print("Libraries ready.")
except ImportError:
    print("Installing RDKit & MDAnalysis...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rdkit", "MDAnalysis"])
    from rdkit import Chem
    from rdkit.Chem import AllChem
    import MDAnalysis as mda

# ==============================================================================
#  CONFIGURATION
# ==============================================================================

# Input Structure (CIF format from Protenix/AlphaFold)
TARGET_MAP = "wt_GSTO1_PP_v3_sample_0.cif"

# Output Filenames
OUTPUT_PDB = "final_system_v5_3.pdb"        # Merged system for MD setup
OUTPUT_PP  = "plastic_final3.mol2"          # Optimized Ligand Topology
OUTPUT_GSH = "gsh_final3.mol2"              # Optimized Cofactor Topology

# ==============================================================================
#  EXECUTION
# ==============================================================================

# 1. Target Site Identification
print(f"Reading target map from: {TARGET_MAP}")

if not os.path.exists(TARGET_MAP):
    print(f"ERROR: Could not find {TARGET_MAP}")
    raise FileNotFoundError("Missing Protenix input file.")

# Convert CIF to PDB for compatibility with MDAnalysis
if not os.path.exists("/usr/bin/obabel"):
    subprocess.run("apt-get install -y openbabel > /dev/null", shell=True)

subprocess.run(f"obabel {TARGET_MAP} -O temp_map.pdb", shell=True)

u = mda.Universe("temp_map.pdb")

# Select Ligand (Chain B) and Cofactor (Chain C) based on standard Protenix output
ref_plastic = u.select_atoms("chainID B or resname LIG or resname UNK")
ref_gsh     = u.select_atoms("chainID C or resname GSH")

# Fallback selection if chain logic fails
if len(ref_plastic) == 0:
    print("Warning: Auto-detection failed. Attempting selection by mass/type...")
    ref_plastic = u.select_atoms("not protein and not resname HOH and mass < 600 and mass > 100")

# Calculate geometric centers for docking alignment
plastic_center = ref_plastic.center_of_mass()
gsh_center     = ref_gsh.center_of_mass()

print(f"Plastic Center of Mass: {plastic_center}")
print(f"GSH Center of Mass:     {gsh_center}")

# 2. Structure Regeneration (Sanitization)
print("Regenerating clean chemical structures...")

# Plastic: Oxidized PP Trimer
PP_SMILES = "CC(C)CC(=O)CC(C)C"
mol_pp = Chem.MolFromSmiles(PP_SMILES)
mol_pp = Chem.AddHs(mol_pp)
AllChem.EmbedMolecule(mol_pp, AllChem.ETKDGv3())

# Cofactor: L-Glutathione
GSH_SMILES = "C(CC(=O)N[C@@H](CS)C(=O)NCC(=O)O)[C@@H](C(=O)O)N"
mol_gsh = Chem.MolFromSmiles(GSH_SMILES)
mol_gsh = Chem.AddHs(mol_gsh)
AllChem.EmbedMolecule(mol_gsh, AllChem.ETKDGv3())

# 3. Coordinate Alignment and Topology Export
def save_matched_files(mol, target_pos, mol2_filename, res_name, chain_id):
    # A. Translation to Target Site
    conf = mol.GetConformer()
    pos = conf.GetPositions()
    current_center = np.mean(pos, axis=0)
    shift = target_pos - current_center
    new_pos = pos + shift

    for i in range(mol.GetNumAtoms()):
        conf.SetAtomPosition(i, new_pos[i])

    # B. Atom Naming Standardization (C1, C2...) for CHARMM-GUI compatibility
    atom_counts = {}
    for atom in mol.GetAtoms():
        sym = atom.GetSymbol()
        atom_counts[sym] = atom_counts.get(sym, 0) + 1
        atom_name = f"{sym}{atom_counts[sym]}" # e.g., C1, C2, O1
        atom.SetProp("_Name", atom_name)

        # Set PDB Residue Info
        info = Chem.AtomPDBResidueInfo()
        info.SetName(f"{atom_name:<4}")
        info.SetResidueName(res_name)
        info.SetResidueNumber(1)
        info.SetChainId(chain_id)
        info.SetIsHeteroAtom(True)
        atom.SetMonomerInfo(info)

    # C. Export to MOL2 via OpenBabel (preserves atom types and charges)
    temp_pdb = f"temp_{res_name}.pdb"
    Chem.MolToPDBFile(mol, temp_pdb)

    cmd = f"obabel -ipdb {temp_pdb} -omol2 -O {mol2_filename} --unique"
    subprocess.run(cmd, shell=True, check=True)

    print(f"Generated topology: {mol2_filename}")
    return temp_pdb

# Generate aligned fragments
pdb_frag_pp  = save_matched_files(mol_pp, plastic_center, OUTPUT_PP, "LIG", "B")
pdb_frag_gsh = save_matched_files(mol_gsh, gsh_center, OUTPUT_GSH, "GSH", "C")

# 4. Final PDB Assembly
print(f"Assembling final system: {OUTPUT_PDB}")

with open(OUTPUT_PDB, "w") as outfile:
    # Write Protein (Chain A)
    u_prot = mda.Universe("temp_map.pdb")
    prot_atoms = u_prot.select_atoms("protein or chainID A")

    # Use MDAnalysis writer to ensure standard PDB formatting
    prot_atoms.write("temp_prot_only.pdb")
    with open("temp_prot_only.pdb", "r") as f:
        for line in f:
            if line.startswith("ATOM"):
                outfile.write(line)

    outfile.write("TER\n")

    # Write Plastic Ligand (Chain B)
    with open(pdb_frag_pp, "r") as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                # Enforce Chain B and Residue LIG
                outfile.write(line[:17] + "LIG B" + line[22:])

    outfile.write("TER\n")

    # Write GSH Cofactor (Chain C)
    with open(pdb_frag_gsh, "r") as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                # Enforce Chain C and Residue GSH
                outfile.write(line[:17] + "GSH C" + line[22:])

    outfile.write("END\n")

# Cleanup temporary files
for f in ["temp_map.pdb", "temp_prot_only.pdb", pdb_frag_pp, pdb_frag_gsh]:
    if os.path.exists(f): os.remove(f)

print("\nProcessing Complete. Generated Files:")
print(f"   1. {OUTPUT_PDB} (Upload to CHARMM-GUI Step 1)")
print(f"   2. {OUTPUT_PP} (Upload to CHARMM-GUI Step 2 for LIG)")
print(f"   3. {OUTPUT_GSH} (Upload to CHARMM-GUI Step 2 for GSH)")